# ATML PA2 — Colab runner (thin wrapper, all code lives in the GitHub repo)

**State is shared across Colab accounts through one Google Drive folder.**

## One-time setup (do this once, from your main Google account)
1. In Google Drive create a folder named **`PA2_state`**.
2. Share it (Editor) with every other Google account you will run Colab from.
3. In each other account: open *Shared with me* → right-click `PA2_state` → **Organize → Add shortcut → My Drive**.

Every account then sees the same path `/content/drive/MyDrive/PA2_state`, which holds:
- `outputs/` — trained LoRA adapters (needed later by Task 4)
- `results/` — manifests, logs, metrics (returned to GitHub)

## Every session
1. Runtime → Change runtime type → **T4 GPU**.
2. Set `WORKER` below to a name for this account (e.g. `acct1`, `acct2`).
3. Run the **Setup** cells, then the experiment cell you want.
4. Look at the **status board** first — never start an experiment that another account shows as `running`.

Each experiment writes `status.json` (running/done/failed). Finished runs are skipped automatically,
a run with a fresh heartbeat on another account is refused, and a crashed run is archived (not overwritten) before retrying.

In [12]:
# ---- Session settings ----
WORKER = "acct1"                      # <-- change per Google account
REPO_URL = "https://github.com/TalhaShoyo10/PA2_28100131.git"
REF = "main"                          # branch or exact commit SHA to run
STATE_DIR = "/content/drive/MyDrive/ATML_PA2"
REPO_DIR = "/content/PA2"

## Setup

In [13]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(STATE_DIR), f"{STATE_DIR} not found - create it or add the shared shortcut to My Drive (see top)."
for sub in ("outputs", "results", "results/logs"):
    os.makedirs(f"{STATE_DIR}/{sub}", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [14]:
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {REF} && git pull -q --ff-only || true
!git log --oneline -1 && git status --short --untracked-files=no

/content/PA2
c526ffb (HEAD -> main, origin/main, origin/HEAD) Fix status-board cell syntax error in Colab runner


In [15]:
!pip install -q -r requirements.txt

In [16]:
# Course assets (~1.07 GB, pinned revision) + validation + environment record + objective tests
!python -m scripts.download_assets
!python -m scripts.validate_assets
!python -m scripts.check_environment
!python -m pytest tests/ -q -p no:cacheprovider

Source: AbDu11aHHH/ATML-PA2-assets
Revision: 0b350481fb03f5525a35bcdec4131bd4fe487f98
Fetching 90 files: 100% 90/90 [00:00<00:00, 937.30it/s]

Course assets downloaded.
Now run: python -m scripts.validate_assets
dpo_standard_train           OK  data/dpo_standard_train.jsonl
dpo_standard_eval            OK  data/dpo_standard_eval.jsonl
dpo_length_train             OK  data/dpo_length_balanced_train.jsonl
dpo_length_eval              OK  data/dpo_length_stratified_eval.jsonl
word_limit_prompts           OK  data/word_limit_prompts.jsonl
rl_prompt_train              OK  data/rl_prompt_pool_train.jsonl
rl_prompt_eval               OK  data/rl_prompt_pool_eval.jsonl
xstest                       OK  data/xstest_safety_prompts.csv
gsm_train                    OK  data/gsm8k_rl_train.jsonl
gsm_eval                     OK  data/gsm8k_eval.jsonl
math_transfer_eval           OK  data/math_transfer_eval.jsonl
task5_diagnostics            OK  data/task5_controlled_reward_diagnostics.jsonl
ppo_rollo

In [ ]:
# outputs/ (adapters, git-ignored) -> symlink to Drive.
# results/ stays the git snapshot; new results are written straight to Drive via PA2_RESULTS_ROOT.
out_path = os.path.join(REPO_DIR, "outputs")
if not os.path.islink(out_path):
    if os.path.isdir(out_path):
        assert not os.listdir(out_path), f"{out_path} has local files; move them to Drive first"
        os.rmdir(out_path)
    os.symlink(f"{STATE_DIR}/outputs", out_path)
res_path = os.path.join(REPO_DIR, "results")
if os.path.islink(res_path):          # left over from an older session layout
    os.remove(res_path)
    !cd {REPO_DIR} && git checkout -- results
os.environ["PA2_RESULTS_ROOT"] = f"{STATE_DIR}/results"
os.environ["PA2_WORKER"] = WORKER
os.environ["PYTHONUNBUFFERED"] = "1"
!ls -la {REPO_DIR} | grep -E "outputs|results"
print("results will be written to", os.environ["PA2_RESULTS_ROOT"])


In [ ]:
import datetime, json, glob, time

def run(cmd, log_name):
    """Run a repo command, stream output, and keep a copy in results/logs on Drive."""
    ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    log = f"{STATE_DIR}/results/logs/{log_name}__{WORKER}__{ts}.log"
    get_ipython().system(f"bash -c 'set -o pipefail; cd {REPO_DIR} && {cmd} 2>&1 | tee {log}'")
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed (exit {code}); log: {log}")

def board():
    """Status of every experiment in the shared Drive state."""
    rows = []
    for p in sorted(glob.glob(f"{STATE_DIR}/results/*/*/status.json")):
        s = json.load(open(p))
        age = (time.time() - s.get("updated_unix", 0)) / 60
        prog = f"{s.get('update', s.get('step', ''))}/{s.get('of', '')}" if s["state"] == "running" else ""
        rows.append(f"{s['state']:8s} {s['experiment_id']:48s} {s.get('worker',''):8s} {age:6.1f} min ago  {prog}")

    for r in rows or ["no experiments yet"]:
        print(r)


board()

## Task 1 — DPO

Order: smoke test → standard (train + eval) → β forks → length study → summary.
Approximate T4 times are estimates, not measurements: smoke ≈ 5 min, standard train ≈ 40–70 min,
each eval ≈ 10–15 min, each β fork ≈ 20 min + eval, length-balanced ≈ same as standard.
Different cells can run on different accounts at the same time.

In [19]:
# Smoke test: 2 optimizer updates + tiny evaluation. Catches crashes before spending an hour.
run("python -m task1_dpo.train --run-name smoke --max-examples 32", "task1_smoke_train")
run("python -m task1_dpo.evaluate --adapter outputs/task1_dpo/smoke --name smoke --limit 4", "task1_smoke_eval")

/tmp/ipykernel_1912/2406323809.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


I0000 00:00:1791327502.270274    5670 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791327505.595676    5670 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[smoke] update 1/2 loss 0.6931 acc 0.000 gnorm 2.302 t 14s
[smoke] update 2/2 loss 0.6908 acc 0.625 gnorm 1.713 t 23s
I0000 00:00:1791327579.656307    6078 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791327582.379948    6078 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[1/4] held-out pairs
[2/4] l

In [ ]:
# Step 1: standard DPO, one epoch from the original initialization (its adapter is the Task 4 DPO policy)
run("python -m task1_dpo.train --config configs/dpo.yaml --run-name standard", "task1_train_standard")

In [ ]:
run("python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard", "task1_eval_standard")

In [ ]:
# Additional reference point (not a required condition): the untouched SFT policy under the same protocol
run("python -m task1_dpo.evaluate --config configs/dpo.yaml --name sft", "task1_eval_sft")

In [ ]:
# Step 2: beta forks — one per cell so they can be split across accounts
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.03", "task1_beta003")

In [ ]:
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.10", "task1_beta010")

In [ ]:
run("python -m task1_dpo.ablate_beta --config configs/dpo.yaml --only 0.30", "task1_beta030")

In [ ]:
# Step 3: length-balanced DPO + comparison (needs the standard eval to be done)
run("python -m task1_dpo.analyze_length --config configs/dpo.yaml", "task1_length")

In [ ]:
run("python -m task1_dpo.summarize --config configs/dpo.yaml", "task1_summary")

## Task 2 — PPO

Order: smoke → cached clipping study → **standard (train + eval, needed by Task 4)** → midpoint eval → clipping forks → KL forks → summary.
Every PPO run starts from the identical supplied midpoint. The ε = 0.20 clip fork and the βKL = 0.10 KL fork have the
same configuration, so they are one shared run (`fork_eps020_kl010`); whichever cell reaches it second skips it.
Times are not measured yet — the smoke run will tell.

In [ ]:
# Smoke test: 2 PPO updates + tiny held-out evaluation
run("python -m task2_ppo.continue_train --run-name smoke --updates 2", "task2_smoke_train")
run("python -m task2_ppo.evaluate --adapter outputs/task2_ppo/smoke --name smoke --limit 4", "task2_smoke_eval")

In [ ]:
# Cached rollout clipping diagnostic (no generation; fixed supplied batch, eps = 0.05 / 0.20 / 0.50)
run("python -m task2_ppo.analyze_clipping --part cached", "task2_cached_clipping")

In [ ]:
# Step 1: standard 20-update continuation (its adapter is the Task 4 PPO policy)
run("python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard", "task2_train_standard")

In [ ]:
run("python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard", "task2_eval_standard")

In [ ]:
# Reference point: the supplied midpoint before any continuation, same evaluation protocol
run("python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter checkpoints/ppo_midpoint_policy --name midpoint", "task2_eval_midpoint")

In [ ]:
# Step 2 forks: one epsilon per cell (beta_KL fixed at 0.10)
run("python -m task2_ppo.analyze_clipping --part forks --only 0.05", "task2_clip005")

In [ ]:
run("python -m task2_ppo.analyze_clipping --part forks --only 0.20", "task2_clip020")

In [ ]:
run("python -m task2_ppo.analyze_clipping --part forks --only 0.50", "task2_clip050")

In [ ]:
# Step 3 forks: one beta_KL per cell (epsilon fixed at 0.20; 0.10 is the shared fork above)
run("python -m task2_ppo.ablate_kl --config configs/ppo.yaml --only 0.0", "task2_kl000")

In [ ]:
run("python -m task2_ppo.ablate_kl --config configs/ppo.yaml --only 0.10", "task2_kl010")

In [ ]:
run("python -m task2_ppo.ablate_kl --config configs/ppo.yaml --only 0.20", "task2_kl020")

In [ ]:
run("python -m task2_ppo.summarize --config configs/ppo.yaml", "task2_summary")

## Task 3 — GRPO

Order: smoke → **standard (train + eval, needed by Task 4)** → midpoint eval → group-size study (no model, seconds) →
normalization forks (one per cell) → summary.

In [ ]:
# Smoke test: 2 GRPO updates (K = 4 completions each) + tiny held-out evaluation
run("python -m task3_grpo.continue_train --run-name smoke --updates 2", "task3_smoke_train")
run("python -m task3_grpo.evaluate --adapter outputs/task3_grpo/smoke --name smoke --limit 4", "task3_smoke_eval")

In [ ]:
# Step 1: standard 20-update continuation (its adapter is the Task 4 GRPO policy)
run("python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard", "task3_train_standard")

In [ ]:
run("python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard", "task3_eval_standard")

In [ ]:
# Reference point: the supplied midpoint before any continuation, same protocol
run("python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter checkpoints/grpo_midpoint_policy --name midpoint", "task3_eval_midpoint")

In [ ]:
# Step 2: group size K = 2 / 4 / 8 at equal total generations, from the supplied cache (no model needed)
run("python -m task3_grpo.analyze_group_size --config configs/grpo.yaml", "task3_group_size")

In [ ]:
# Step 3: canonical GRPO normalization fork
run("python -m task3_grpo.compare_normalization --config configs/grpo.yaml --only grpo", "task3_norm_grpo")

In [ ]:
# Step 3: Dr-GRPO normalization fork
run("python -m task3_grpo.compare_normalization --config configs/grpo.yaml --only dr_grpo", "task3_norm_drgrpo")

In [ ]:
run("python -m task3_grpo.summarize --config configs/grpo.yaml", "task3_summary")

## Task 4 — Safety calibration

Needs the finished **standard** DPO, PPO and GRPO runs (the code refuses to start otherwise; ablation forks are never used).
1. Generate greedy responses to all 450 XSTest prompts — one cell per policy (can be split across accounts).
2. Run the fixed AI judge — one cell per policy.
3. Make the blind audit sheet (240 items = 60 fixed prompts x 4 policies).
4. Evaluate (rates now; agreement after you have filled in your manual labels).

**Do not open `judged_*.jsonl` or `audit_key.csv` before you finish labelling.**

In [ ]:
run("python -m task4_safety.generate_responses --policy sft", "task4_generate_sft")

In [ ]:
run("python -m task4_safety.generate_responses --policy dpo", "task4_generate_dpo")

In [ ]:
run("python -m task4_safety.generate_responses --policy ppo", "task4_generate_ppo")

In [ ]:
run("python -m task4_safety.generate_responses --policy grpo", "task4_generate_grpo")

In [ ]:
run("python -m task4_safety.judge_responses --policy sft", "task4_judge_sft")

In [ ]:
run("python -m task4_safety.judge_responses --policy dpo", "task4_judge_dpo")

In [ ]:
run("python -m task4_safety.judge_responses --policy ppo", "task4_judge_ppo")

In [ ]:
run("python -m task4_safety.judge_responses --policy grpo", "task4_judge_grpo")

In [ ]:
# Blind audit sheet -> Drive: ATML_PA2/results/task4_safety/audit_sheet_blind.csv
run("python -m task4_safety.make_audit_sheet", "task4_audit_sheet")

In [ ]:
# Rates + category tables now; manual agreement once manual_audit_labels.csv is in the same folder
run("python -m task4_safety.evaluate_safety", "task4_evaluate")

## Return results to GitHub
Zips `results/` (small JSON/CSV/log files only — adapters stay in `outputs/` on Drive) and downloads it.
Unzip it into the repo's `results/` folder on your PC and ask Claude to validate and commit.

In [ ]:
import shutil
from google.colab import files
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
archive = shutil.make_archive(f"/content/pa2_results_{stamp}", "zip", f"{STATE_DIR}/results")
print(archive, round(os.path.getsize(archive) / 2**20, 2), "MB")
files.download(archive)